In [6]:
import pandas as pd
pd.set_option('display.max_colwidth', 150)
import numpy as np
from ordered_set import OrderedSet
from sklearn.metrics.pairwise import cosine_similarity
import requests

# Data Prep

## SDG Indicators

Source: https://unstats.un.org/sdgs/indicators/indicators-list/

In [7]:
# # Ingest + Prep SDG Corpus
# SDGS = pd.read_excel('sdg_indicators.xlsx')
# SDGS['SDG'] = SDGS['Target'].apply(lambda x: x if str(x).startswith('Goal') else np.nan).ffill()
# SDGS['Target'] = SDGS['Target'].ffill()
# SDGS = SDGS[~SDGS.Target.apply(lambda x: x.startswith('Goal'))]
# SDGS = SDGS.dropna().drop_duplicates().reset_index(drop = True)
# SDGS.insert(0, 'Target No.', SDGS.Target.apply(lambda target: target.split()[0]))
# SDGS.insert(0, 'SDG No.', SDGS.SDG.apply(lambda goal: int(goal.split('.')[0].replace('Goal', '').strip())))
# SDGS.to_csv('sdg_indicators_corpus.csv', index = False)


SDGS = pd.read_csv('sdg_indicators_corpus.csv')

"""
SDG text aggregation setup
"""
SDGS = SDGS.groupby(['SDG No.', 'Target No.', 'SDG', 'Target'])['Indicator'].apply(lambda x: '\n'.join(x)).reset_index().reset_index(drop = True).rename(columns = {'Indicator' : 'Indicators'})

## Legiscan

https://legiscan.com/gaits/documentation/legiscan




Note on statuses:
- 1 = Introduced
- 2 = Held
- 3 = Sent to Executive Branch
- 4 = Passed
- 5 = Vetoed
- 6 = Died

Probably will want to just look at those passed.


In [140]:
apikey = '3450b965641ca5cf68d314cc3e375ad0'
year = '2026'

dataset_list_url = f'https://api.legiscan.com/?key={apikey}&op=getDatasetList&year={year}'
response = requests.get(dataset_list_url)



In [147]:
legiscan_api_key = '3450b965641ca5cf68d314cc3e375ad0'
session_id = '2218'
access_key = '5IjbjbeAI1TauMILZCDQ8a'

dataset_url = f'https://api.legiscan.com/?key={apikey}&op=getDataset&id={session_id}&access_key={access_key}'
response = requests.get(dataset_url)

In [148]:
import base64
import io
import zipfile
import json


data = {}
b64_zip = response.json()['dataset']['zip']
zip_bytes = base64.b64decode(b64_zip)
with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
    bill_files = [f for f in z.namelist() if '/bill/' in f]
    for bill_file in bill_files:
        with z.open(bill_file) as f:
            data[bill_file] = json.load(f)


In [149]:
len(data.keys())

932

In [150]:
data

{'AL/2026-2026_Regular_Session/bill/HB1.json': {'bill': {'bill_id': 2038126,
   'change_hash': '52ea1e0b9dd13f81ffd4d1133258e195',
   'session_id': 2218,
   'session': {'session_id': 2218,
    'state_id': 1,
    'year_start': 2026,
    'year_end': 2026,
    'prefile': 0,
    'sine_die': 0,
    'prior': 0,
    'special': 0,
    'session_tag': 'Regular Session',
    'session_title': '2026 Regular Session',
    'session_name': '2026 Regular Session'},
   'url': 'https://legiscan.com/AL/bill/HB1/2026',
   'state_link': 'https://alison.legislature.state.al.us/bill-search',
   'completed': 0,
   'status': 2,
   'status_date': '2026-02-03',
   'progress': [{'date': '2026-01-13', 'event': 1},
    {'date': '2026-01-13', 'event': 9},
    {'date': '2026-01-21', 'event': 10},
    {'date': '2026-02-03', 'event': 2}],
   'state': 'AL',
   'state_id': 1,
   'bill_number': 'HB1',
   'bill_type': 'B',
   'bill_type_id': '1',
   'body': 'H',
   'body_id': 11,
   'current_body': 'H',
   'current_body_id'

In [3]:
def get_bills(apikey, legislative_bodies):

    # Iterate states
    BILLS = []
    for stateid in legislative_bodies:

        try:
            print(f'Fetching {stateid}...')

            # Fetch bill data for state
            url = 'https://api.legiscan.com/?key={}&op=getMasterList&state={}'
            response = requests.get(url.format(apikey, stateid))
            response = response.json()['masterlist']

            # Iterate bills and add rows to dataframe
            for i in range(len(response) - 1):
                bill = response[str(i)]
                BILLS.append([stateid, bill['number'], bill['status'], bill['status_date'], bill['title'], bill['description'], bill['url'], bill['last_action']])
        except:
            print(f'Could not fetch {stateid}')

    # Add bills to dataframe
    BILLS = pd.DataFrame(columns = ['legislative_body', 'bill_number', 'status', 'status_date', 'title', 'description', 'legiscan_url', 'last_action'], data = BILLS).fillna('')
    BILLS['status_date'] = pd.to_datetime(BILLS['status_date'])

    # Map status codes & filter to passed / signed legislation
    statuses = {1 : 'Introduced', 2 : 'Held', 3 : 'Passed', 4 : 'Signed', 5 : 'Vetoed', 6 : 'Died'}
    BILLS['status'] = BILLS.status.apply(lambda x: statuses[x] if x in statuses.keys() else 'Other')

    # Adding a UUID collumn
    BILLS.insert(0, 'UUID', BILLS.apply(lambda row: '-'.join([row['legislative_body'], row['bill_number']]), axis = 1))

    return BILLS

apikey = '3450b965641ca5cf68d314cc3e375ad0'


legislative_bodies = state_abbreviations = [
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "FL", "GA",
    "HI", "ID", "IL", "IN", "IA", "KS", "KY", "LA", "ME", "MD",
    "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV", "NH", "NJ",
    "NM", "NY", "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC",
    "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV", "WI", "WY",
    "DC", "US"
]


"""
TEMP - US and LA only
"""

# legislative_bodies = ['US', 'LA']

# BILLS = get_bills(apikey, legislative_bodies)
# BILLS.to_csv('BILLS.csv', index = False)

BILLS = pd.read_csv('BILLS.csv')



In [6]:
apikey = '3450b965641ca5cf68d314cc3e375ad0'
stateid = 'NY'

url = 'https://api.legiscan.com/?key={}&op=getMasterList&state={}'
response = requests.get(url.format(apikey, stateid))
response = response.json()['masterlist']

In [7]:
response

{'session': {'session_id': 2188,
  'state_id': 32,
  'year_start': 2025,
  'year_end': 2026,
  'prefile': 0,
  'sine_die': 0,
  'prior': 0,
  'special': 0,
  'session_tag': 'Regular Session',
  'session_title': '2025-2026 Regular Session',
  'session_name': '2025-2026 General Assembly'},
 '0': {'bill_id': 1898080,
  'number': 'A00021',
  'change_hash': 'ea5e5ed5eb8889e59186c12405275548',
  'url': 'https://legiscan.com/NY/bill/A00021/2025',
  'status_date': '2025-01-08',
  'status': 1,
  'last_action_date': '2026-01-07',
  'last_action': 'referred to insurance',
  'title': 'Requires insurance policies to provide coverage for services related to the diagnosis and treatment of mental, nervous or emotional disorders or ailments; makes related provisions.',
  'description': 'Requires insurance policies to provide coverage for services related to the diagnosis and treatment of mental, nervous or emotional disorders or ailments; makes related provisions.'},
 '1': {'bill_id': 1898054,
  'numbe

# Embedding setup

In [17]:
"""
Generate embeddings for SDGS and Bill titles / descriptions
"""

from sentence_transformers import SentenceTransformer
model = SentenceTransformer('sentence-transformers/all-MiniLM-L12-v1')

SDG_embeddings = model.encode(SDGS.apply(lambda row: '\n'.join(list(OrderedSet([row['Target'].strip(), row['Indicators'].strip()]))), axis = 1))
SDGS['embedding'] = list(SDG_embeddings)

BILL_embeddings = model.encode(BILLS.apply(lambda row: '\n'.join(list(OrderedSet([row['title'].strip(), row['description'].strip()]))), axis = 1))
BILLS['embedding'] = list(BILL_embeddings)


# Tagging

In [76]:
def get_sdg_tags(bill_embedding, SDGS = SDGS, threshold = 0.25):

    # Copy SDGS table & calculate similarities
    SDGS_temp = SDGS.copy()
    SDGS_temp['similarity'] = SDGS_temp.embedding.apply(lambda sdg_embedding: 
                                                        cosine_similarity([bill_embedding], [sdg_embedding])[0][0]
                                                        )
    
    # Filter out below-treshold similarity & return top 3 matched targets
    SDGS_temp = SDGS_temp[SDGS_temp.similarity >= threshold].sort_values(by = 'similarity', ascending=False).head(3).reset_index(drop = True)

    # Match payload
    SDG_tags = {'sdgs': list(set(SDGS_temp['SDG No.'])), 'targets': list(set(SDGS_temp['Target No.']))}
    return SDG_tags


def breakout_sdg_tags(BILLS):

    BILLS_TEMP = pd.DataFrame(columns = list(BILLS.drop(columns = 'sdg_tags').columns) + ['sdg_num', 'sdg_name', 'target_num', 'target_name'])
    sdg_dict = SDGS[['SDG No.', 'SDG']].drop_duplicates().set_index('SDG No.')['SDG'].to_dict()
    target_dict = SDGS[['Target No.', 'Target']].drop_duplicates().set_index('Target No.')['Target'].to_dict()

    for row in BILLS.itertuples():
        if len(row.sdg_tags['sdgs']) == 0:
            new_row = [row.UUID, row.legislative_body, row.bill_number, row.status, row.status_date, row.title, row.description, row.legiscan_url, row.last_action, None, None, None, None]
            BILLS_TEMP.loc[len(BILLS_TEMP)] = new_row
        else:
            for sdg in row.sdg_tags['sdgs']:
                for target in row.sdg_tags['targets']:
                    if target.startswith(str(sdg)):
                        new_row = [row.UUID, row.legislative_body, row.bill_number, row.status, row.status_date, row.title, row.description, row.legiscan_url, row.last_action, sdg, sdg_dict[sdg], target, target_dict[target]]
                        BILLS_TEMP.loc[len(BILLS_TEMP)] = new_row

    return BILLS_TEMP


# # Apply tags
# BILLS['sdg_tags'] = BILLS.embedding.apply(lambda embedding: get_sdg_tags(embedding))

# # Breakout tags
# BILLS = breakout_sdg_tags(BILLS)

# # Save to CSV
# BILLS.to_csv('BILLS_TAGGED.csv', index = False)
